In [187]:

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer, util
from rapidfuzz import fuzz
import ast


In [188]:
# --- 1. GLOBAL SETUP (Run Once) ---
nltk.download('stopwords', quiet=True)
# Keep critical "logic" words that standard NLTK removes
STOP_WORDS = set(stopwords.words('english')) - {'with', 'for', 'above', 'below', 'under', 'over', 'before', 'after'}
MODEL = SentenceTransformer('all-MiniLM-L6-v2') # Optimized for M1

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4974.48it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [189]:
# --- 2. CLEANING LOGIC ---
def clean_market_text(text):
    if pd.isna(text) or text == "":
        return ""
    
    # Lowercase & Currency Normalization
    text = str(text).lower()
    text = text.replace('$', ' usd ').replace('£', ' gbp ').replace('€', ' eur ')
    
    # Strip web-scraped artifacts & marketing fluff
    text = re.sub(r'<(.*?)>', '', text) # Remove HTML
    fluff = ["best deal", "buy now", "limited offer", "free shipping", "on sale"]
    for phrase in fluff:
        text = text.replace(phrase, "")
    
    # Keep alphanumeric and critical punctuation for prices (like decimals)
    text = re.sub(r'[^a-z0-9\s\.]', '', text)
    
    # Final cleanup: Remove stopwords & extra whitespace
    words = text.split()
    return " ".join([w for w in words if w not in STOP_WORDS])

def extract_numbers(text):
    """Utility to pull numbers/prices for a hard-logic check."""
    return set(re.findall(r'\d+\.?\d*', str(text)))

In [197]:

# --- 3. DATA TRANSFORMERS ---
def transform_kalshi(df):
    unified = pd.DataFrame(index=df.index)
    unified['platform'] = 'kalshi'
    unified['market_id'] = df['market_id']
    unified['tags'] = df['tags'].apply(lambda x: [x.lower()] if isinstance(x, str) else [])
    
    # Structural Text
    unified['event_id'] = df['event_id']
    unified['event_title'] = df['event_title']
    unified['market_subtitle'] = df['market_subtitle']
    unified['description'] = df['description'].fillna('')
    
    # Pricing (0.0 - 1.0)
    unified['yes_bid'] = df['yes_bid'].astype(float)
    unified['yes_ask'] = df['yes_ask'].astype(float)
    return unified

def transform_polymarket(df):
    unified = pd.DataFrame(index=df.index)
    unified['platform'] = 'polymarket'
    unified['market_id'] = df['market_id'].astype(str)
    
    # Complex Tag Parsing
    def parse_tags(val):
        try:
            if isinstance(val, str) and val.startswith('['):
                val = ast.literal_eval(val)
            return [str(t).lower().strip() for t in val] if isinstance(val, list) else []
        except: return []
    
    unified['tags'] = df['tags'].apply(parse_tags)
    
    # Structural Text
    unified['event_id'] = df['id']
    unified['event_title'] = df['title']
    unified['market_subtitle'] = df['question']
    unified['description'] = df['description'].fillna('')
    
    # Pricing (Normalize 0.0 - 1.0)
    unified['yes_bid'] = pd.to_numeric(df['bestBid'], errors='coerce')
    unified['yes_ask'] = pd.to_numeric(df['bestAsk'], errors='coerce')
    return unified

In [191]:
# --- 4. SEMANTIC PIPELINE ---
def get_event_matches(df_poly, df_kalshi, tag_filter='elections'):
    """
    Identifies if the high-level EVENTS match between platforms.
    Ignore individual market 'Yes/No' questions for now.
    """
    # 1. Deduplicate to Event Level
    # We only care about unique Event Titles within each platform
    p_events = df_poly.explode('tags').query(f"tags == '{tag_filter}'")
    p_events = p_events[['event_id', 'event_title', 'tags']].drop_duplicates(subset=['event_id'])
    
    k_events = df_kalshi.explode('tags').query(f"tags == '{tag_filter}'")
    k_events = k_events[['event_id', 'event_title', 'tags']].drop_duplicates(subset=['event_id'])

    # 2. Cross-Join on Tags
    pairs = pd.merge(p_events, k_events, on='tags', suffixes=('_poly', '_kalshi'))
    
    if pairs.empty:
        return pairs

    # 3. Simple Text Prep (Event Title Only)
    # We clean just the event titles to keep it lightweight
    pairs['event_title_poly_clean'] = pairs['event_title_poly'].map(clean_market_text)
    pairs['event_title_kalshi_clean'] = pairs['event_title_kalshi'].map(clean_market_text)

    # 4. Semantic Comparison
    print(f"Comparing {len(pairs)} Unique Event Titles...")
    
    # Vectorize cleaned titles
    p_embs = MODEL.encode(pairs['event_title_poly_clean'].tolist(), convert_to_numpy=True)
    k_embs = MODEL.encode(pairs['event_title_kalshi_clean'].tolist(), convert_to_numpy=True)
    
    # Normalize for dot product (Cosine Similarity)
    p_embs = p_embs / np.linalg.norm(p_embs, axis=1, keepdims=True)
    k_embs = k_embs / np.linalg.norm(k_embs, axis=1, keepdims=True)
    
    # Calculate Similarity
    pairs['event_similarity'] = np.sum(p_embs * k_embs, axis=1)

    # 5. Fuzzy "Fail-safe"
    # This helps catch if names are just flipped (e.g. "Kansas Governor" vs "Governor of Kansas")
    def fuzzy_check(row):
        return fuzz.token_sort_ratio(row['event_title_poly_clean'], row['event_title_kalshi_clean']) / 100.0

    pairs['fuzzy_score'] = pairs.apply(fuzzy_check, axis=1)
    
    # Combined Event Score
    pairs['final_event_score'] = (pairs['event_similarity'] * 0.8) + (pairs['fuzzy_score'] * 0.2)

    # Cleanup memory
    cols_to_drop = ['event_title_poly_clean', 'event_title_kalshi_clean']
    return pairs.drop(columns=cols_to_drop).sort_values(by='final_event_score', ascending=False)


In [198]:
# --- 5. EXECUTION ---
# Load Data
df_p_raw = pd.read_csv("data/polymarket_master_data.csv")
df_k_raw = pd.read_csv('data/kalshi_master_data.csv')

# Transform
clean_poly = transform_polymarket(df_p_raw)
clean_kalshi = transform_kalshi(df_k_raw)

# Match
matches = get_event_matches(clean_poly, clean_kalshi, tag_filter='elections')

# Display Top Results

KeyError: 'tags'

In [ ]:
# matches[matches['event_similarity'] > 0.70].to_csv('data/arb_candidates.csv', index=False)
# matches[matches['event_similarity'] > 0.60]
matches[matches['event_id_kalshi'] == 'GOVPARTYKS-27']


,event_id_poly,event_title_poly,tags,event_id_kalshi,event_title_kalshi,event_similarity,fuzzy_score,final_event_score
